In [ ]:
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging
from pathlib import Path

from drzavni_posao import (
    MediaStore,
    MlflowTracker,
    PipelineConfig,
    Stage,
    TranscriberConfig,
    run_pipeline,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.pipeline")


In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
TRANSCRIPT_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)

DEVICE = "cuda"
COMPUTE_TYPE = "bfloat16"
ASR_MODEL = "large-v3"
ALIGN_MODEL = "classla/wav2vec2-xls-r-parlaspeech-hr"
INITIAL_CHUNK_SIZE = 20
RUN_NAME = "large-v3-initial-chunk-size-20"

CAST_HOTWORDS = (
    "Đorđe Čvarkov, Dragan Torbica, Boškić, Dobrosavljev, "
    "Marijanović, Živan Radosavljević, Frau Šilovička"
)
ASR_OPTIONS = {
    "no_repeat_ngram_size": 5,
    "repetition_penalty": 1.0,
    "hotwords": CAST_HOTWORDS,
}

tracker = MlflowTracker(EXPERIMENT_NAME)


In [ ]:
media = MediaStore(
    AUDIO_PATHS,
    TRANSCRIPT_PATH,
    trim_start=25,
    trim_end=7,
)

config = PipelineConfig.from_media(
    media,
    transcription=TranscriberConfig(
        device=DEVICE,
        compute_type=COMPUTE_TYPE,
        model_name=ASR_MODEL,
        language="sr",
        align_model_name=ALIGN_MODEL,
        asr_options=ASR_OPTIONS,
        chunk_size=INITIAL_CHUNK_SIZE,
    ),
)

ALL_EPISODES = {episode.key for episode in media.episodes()}
len(ALL_EPISODES), config.generated_name()


In [ ]:
RUN_ID = run_pipeline(
    tracker,
    media,
    config,
    ALL_EPISODES,
    through=Stage.TRANSCRIPTION,
    name=RUN_NAME,
    tags={"initial_chunk_size": str(INITIAL_CHUNK_SIZE)},
)
RUN_ID
